In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas pyyaml gitpython tqdm scikit-learn wandb evaluate
!pip install -q "datasets[audio]" "transformers[torch]" accelerate bitsandbytes scipy
!pip install -q ctranslate2 faster-whisper py-rover peft trl
!pip install -q vllm

In [ ]:
import os
import sys
import torch
import yaml
import argparse
import pandas as pd
import re
import subprocess
import git
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm.notebook import tqdm  # Используем tqdm для ноутбуков

# ASR & Data
import evaluate
from datasets import load_dataset, Audio, Dataset

# Whisper
import ctranslate2
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2SeqWithPadding
)
from faster_whisper import WhisperModel as FasterWhisperModel

# Silero
# (используется через subprocess)

# LLM & Fine-tuning
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
from trl import DPOTrainer
from vllm import LLM, SamplingParams

# Ensemble
from rover import Main as RoverMain

# MLOps
import wandb

print("Все библиотеки успешно импортированы.")

In [ ]:
CONFIG_STR = """
pipeline_control:
  run_data_prep: true
  run_llm_dpo_data_prep: true
  run_whisper_training: true
  run_silero_training: true
  run_llm_finetuning: true
  run_model_merging: true
  run_evaluation: true
  
wandb:
  project: "advanced-asr-corrector-notebook"
  entity: "YOUR_WANDB_USERNAME" # !!! ЗАМЕНИТЕ НА ВАШ ЛОГИН В WANDB !!!

paths:
  source_csv: "clear_rez_transcriptions.csv"
  audio_directory: "./audio_data/" # !!! Убедитесь, что папка существует и содержит аудио !!!
  checkpoints_dir: "./.checkpoints"
  prepared_data_dir: "./prepared_data"
  llm_dpo_data_path: "./prepared_data/llm_dpo_dataset"
  
  whisper_finetuned_path: "./models/whisper-finetuned"
  whisper_ct2_path: "./models/whisper-finetuned-ct2"
  silero_finetuned_path: "./models/silero-finetuned"
  llm_base_model_path: "./models/yagpt-base"
  llm_finetuned_adapter_path: "./models/yagpt-8b-lora-adapter"
  llm_finetuned_merged_path: "./models/yagpt-8b-finetuned-merged"

data_prep:
  path_column: "Путь к файлу"
  text_column: "Расшифровка"
  test_size: 0.1

whisper_params:
  base_model: "openai/whisper-large-v3"
  num_proc: 8
  training_args:
    output_dir: "./models/whisper-finetuned"
    bf16: true 
    per_device_train_batch_size: 16
    gradient_accumulation_steps: 2
    learning_rate: 1.0e-5
    warmup_steps: 200
    max_steps: 2000
    evaluation_strategy: "steps"
    save_strategy: "steps"
    predict_with_generate: true
    save_steps: 500
    eval_steps: 500
    logging_steps: 25
    load_best_model_at_end: true
    metric_for_best_model: "wer"
    greater_is_better: false
    report_to: "wandb"

silero_params:
  repo_url: "https://github.com/snakers4/silero-models.git"
  repo_path: "./silero-models"
  base_model_url: "https://models.silero.ai/models/jit/v4_ru_jit.pt"
  training_params:
    work_dir: "work_dir"
    batch_size: 64
    num_epochs: 20
    learning_rate: 0.0001
    model: {in_channels: 1, out_channels: 1024}

yandex_gpt_params:
  base_model: "yandex-datasphere/yandex-gpt-5-lite-8b-pretrain"
  max_seq_length: 1024
  dpo_prompt_template: "Исправь ошибки в следующей транскрипции: {instruction}"
  
  lora_config:
    r: 16
    lora_alpha: 32
    lora_dropout: 0.05
    bias: "none"
    task_type: "CAUSAL_LM"
    target_modules: ["q_proj", "v_proj", "k_proj", "o_proj"]
  
  dpo_training_args:
    output_dir: "./models/yagpt-8b-lora-adapter-temp" # Временный путь
    per_device_train_batch_size: 2
    gradient_accumulation_steps: 4
    learning_rate: 2.0e-5
    num_train_epochs: 2
    logging_steps: 10
    save_strategy: "epoch"
    bf16: true
    report_to: "wandb"
    
vllm_params:
  tensor_parallel_size: 2
"""

config = yaml.safe_load(CONFIG_STR)
checkpoint_dir = Path(config['paths']['checkpoints_dir'])
checkpoint_dir.mkdir(exist_ok=True)

# Логинимся в W&B
wandb.login()

In [ ]:
def run_stage(stage_name, function, *args, **kwargs):
    """Обертка для запуска этапов с проверкой чекпоинтов."""
    checkpoint_file = checkpoint_dir / f"{stage_name}.done"
    if config['pipeline_control'][f'run_{stage_name}']:
        if checkpoint_file.exists():
            print(f"--- Пропуск этапа '{stage_name}' (найден чекпоинт) ---")
            return
        
        print(f"\n{'='*20} ЗАПУСК ЭТАПА: {stage_name.upper()} {'='*20}")
        function(config, *args, **kwargs)
        checkpoint_file.touch()
        print(f"--- Этап '{stage_name}' успешно завершен ---")
    else:
        print(f"--- Пропуск этапа '{stage_name}' (отключено в конфиге) ---")

def normalize_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_rover_on_pair(hyp1, hyp2, uid="utt"):
    hyp_files = []
    try:
        for i, hyp in enumerate([hyp1, hyp2]):
            filename = f"temp_hyp_{i}.txt"
            with open(filename, 'w', encoding='utf-8') as f: f.write(f"{hyp} ({uid})\n")
            hyp_files.append(filename)
        RoverMain.main(['-h', hyp_files[0], '-h', hyp_files[1], '-m', 'consensus', '-o', 'temp_out.txt'])
        with open('temp_out.txt', 'r', encoding='utf-8') as f: result_line = f.readline()
        return result_line.split(' (', 1)[0].strip()
    finally:
        for f in hyp_files + ['temp_out.txt']:
            if os.path.exists(f): os.remove(f)

print("Вспомогательные функции определены.")

In [ ]:
def prepare_asr_data_func(config):
    paths_cfg = config['paths']
    data_cfg = config['data_prep']
    wandb_cfg = config['wandb']
    
    df = pd.read_csv(paths_cfg['source_csv'])
    new_audio_dir = Path(paths_cfg['audio_directory'])
    df['corrected_path'] = df[data_cfg['path_column']].apply(lambda p: str(new_audio_dir / Path(p).name))
    df['normalized_text'] = df[data_cfg['text_column']].apply(normalize_text)
    df = df.dropna(subset=['normalized_text', 'corrected_path'])
    df = df[df['normalized_text'].str.len() > 0]
    
    print("Проверка существования аудиофайлов...")
    df['exists'] = df['corrected_path'].apply(lambda p: Path(p).exists())
    initial_count = len(df)
    df = df[df['exists']]
    print(f"Найдено {len(df)} из {initial_count} валидных аудиофайлов.")
    
    train_df, test_df = train_test_split(df, test_size=data_cfg['test_size'], random_state=42)
    
    prepared_dir = Path(paths_cfg['prepared_data_dir'])
    prepared_dir.mkdir(exist_ok=True)
    
    # Подготовка для Whisper
    whisper_train_path = prepared_dir / "whisper_train.jsonl"
    whisper_test_path = prepared_dir / "whisper_test.jsonl"
    train_df.rename(columns={'corrected_path': 'audio', 'normalized_text': 'transcription'})[['audio', 'transcription']].to_json(
        whisper_train_path, orient='records', lines=True, force_ascii=False)
    test_df.rename(columns={'corrected_path': 'audio', 'normalized_text': 'transcription'})[['audio', 'transcription']].to_json(
        whisper_test_path, orient='records', lines=True, force_ascii=False)
    
    # Подготовка для Silero
    silero_data_dir = prepared_dir / "silero"
    silero_data_dir.mkdir(exist_ok=True)
    def write_manifest(dataframe, path):
        with open(path, 'w', encoding='utf-8') as f:
            for _, row in dataframe.iterrows():
                abs_path = Path(row['corrected_path']).resolve()
                f.write(f"{abs_path}|{row['normalized_text']}\n")
    write_manifest(train_df, silero_data_dir / 'train.txt')
    write_manifest(test_df, silero_data_dir / 'test.txt')
    
    all_text = "".join(train_df['normalized_text'].tolist())
    unique_chars = sorted(list(set(all_text)))
    with open(silero_data_dir / 'chars.txt', 'w', encoding='utf-8') as f:
        f.write('_\n' + '\n'.join(unique_chars))
    
    print("Логирование датасета в W&B Artifacts...")
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="data_preparation")
    artifact = wandb.Artifact('asr_datasets', type='dataset')
    artifact.add_dir(str(prepared_dir))
    run.log_artifact(artifact)
    run.finish()

run_stage('data_prep', prepare_asr_data_func)

In [ ]:
def prepare_llm_dpo_data_func(config):
    paths_cfg = config['paths']
    llm_cfg = config['yandex_gpt_params']
    wandb_cfg = config['wandb']
    
    train_df = pd.read_json(Path(paths_cfg['prepared_data_dir']) / "whisper_train.jsonl", lines=True)
    
    print("Генерация 'плохих' транскрипций с базовой моделью Whisper...")
    base_whisper = FasterWhisperModel(config['whisper_params']['base_model'], device="cuda", compute_type="float16")
    
    dpo_data = []
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Создание DPO пар"):
        segments, _ = base_whisper.transcribe(row['audio'], language="ru")
        rejected_response = "".join(s.text for s in segments).strip()
        
        dpo_data.append({
            "prompt": llm_cfg['dpo_prompt_template'].format(instruction=rejected_response),
            "chosen": row['transcription'],
            "rejected": rejected_response,
        })
    
    dpo_dataset = Dataset.from_list(dpo_data)
    dpo_dataset.save_to_disk(paths_cfg['llm_dpo_data_path'])
    
    print("Логирование DPO датасета в W&B Artifacts...")
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="dpo_data_prep")
    artifact = wandb.Artifact('dpo_dataset', type='dataset')
    artifact.add_dir(paths_cfg['llm_dpo_data_path'])
    run.log_artifact(artifact)
    run.finish()
    
run_stage('llm_dpo_data_prep', prepare_llm_dpo_data_func)

In [ ]:
def run_whisper_finetune_func(config):
    w_cfg = config['whisper_params']
    paths_cfg = config['paths']
    wandb_cfg = config['wandb']
    
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="train_whisper")
    
    print("Загрузка ASR датасета из W&B Artifacts...")
    artifact = run.use_artifact('asr_datasets:latest')
    artifact_dir = Path(artifact.download())
    
    dataset = load_dataset("json", data_files={
        'train': str(artifact_dir / "whisper_train.jsonl"),
        'test': str(artifact_dir / "whisper_test.jsonl")
    }).cast_column("audio", Audio(sampling_rate=16000))
    
    processor = WhisperProcessor.from_pretrained(w_cfg['base_model'], language="russian", task="transcribe")
    def prepare_fn(batch):
        batch["input_features"] = processor.feature_extractor(batch["audio"]["array"], sampling_rate=16000).input_features[0]
        batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
        return batch
    tokenized_dataset = dataset.map(prepare_fn, remove_columns=list(dataset["train"].features), num_proc=w_cfg['num_proc'])
    
    data_collator = DataCollatorForSeq2SeqWithPadding(processor=processor)
    metric = evaluate.load("wer")
    def compute_metrics(pred):
        pred_ids = pred.predictions; label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        wer = 100 * metric.compute(predictions=[normalize_text(s) for s in pred_str], references=[normalize_text(s) for s in label_str])
        return {"wer": wer}

    model = WhisperForConditionalGeneration.from_pretrained(w_cfg['base_model'])
    model.config.forced_decoder_ids = None
    
    training_args = Seq2SeqTrainingArguments(**w_cfg['training_args'])
    
    trainer = Seq2SeqTrainer(
        args=training_args, model=model, train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"], data_collator=data_collator,
        compute_metrics=compute_metrics, tokenizer=processor.feature_extractor
    )
    
    print("Запуск обучения Whisper...")
    trainer.train()
    print("Обучение Whisper завершено!")
    
    finetuned_path = Path(paths_cfg['whisper_finetuned_path'])
    finetuned_path.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(finetuned_path))
    processor.save_pretrained(str(finetuned_path))
    
    print("Конвертация дообученной модели Whisper в формат CTranslate2...")
    ct2_path = paths_cfg['whisper_ct2_path']
    converter = ctranslate2.converters.TransformersConverter(str(finetuned_path))
    converter.convert(ct2_path, quantization="float16" if torch.cuda.is_available() else "int8", force=True)
    
    print("Логирование модели Whisper в W&B Artifacts...")
    model_artifact = wandb.Artifact(f"whisper-{run.id}", type="model")
    model_artifact.add_dir(ct2_path)
    run.log_artifact(model_artifact)
    run.finish()

# ВНИМАНИЕ: Для запуска на 2-х GPU нужен accelerate launch из терминала
run_stage('whisper_training', run_whisper_finetune_func)

In [ ]:
def run_silero_finetune_func(config):
    s_cfg = config['silero_params']
    paths_cfg = config['paths']
    wandb_cfg = config['wandb']
    
    print("Проверка и настройка окружения Silero...")
    repo_path = Path(s_cfg['repo_path'])
    if not repo_path.exists():
        print(f"Клонирование репозитория Silero из {s_cfg['repo_url']}...")
        git.Repo.clone_from(s_cfg['repo_url'], repo_path)
    
    base_model_path = repo_path / Path(s_cfg['base_model_url']).name
    if not base_model_path.exists():
        print(f"Скачивание базовой модели Silero: {base_model_path.name}...")
        urlretrieve(s_cfg['base_model_url'], base_model_path)
        
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="setup_silero_train")
    artifact = run.use_artifact('asr_datasets:latest')
    data_dir = Path(artifact.download())
    prepared_data_dir = data_dir / "silero"
    
    silero_training_config = s_cfg['training_params'].copy()
    silero_training_config.update({
        'train_path': str((prepared_data_dir / 'train.txt').resolve()),
        'val_path': str((prepared_data_dir / 'test.txt').resolve()),
        'chars_path': str((prepared_data_dir / 'chars.txt').resolve()),
        'load_path': str(base_model_path.resolve()),
        'work_dir': str(Path(paths_cfg['silero_finetuned_path']).resolve()),
    })
    
    silero_config_path = repo_path / 'generated_notebook_config.yaml'
    with open(silero_config_path, 'w', encoding='utf-8') as f:
        yaml.dump(silero_training_config, f, default_flow_style=False, allow_unicode=True)
    run.finish()

    print("\nЗапуск процесса дообучения Silero...")
    train_script_path = repo_path / 'train.py'
    relative_config_dir = silero_config_path.parent.relative_to(repo_path)
    config_name = silero_config_path.stem
    command = [
        "python", str(train_script_path), 
        f"--config-path={relative_config_dir}", 
        f"--config-name={config_name}"
    ]
    
    print(f"Выполнение команды: {' '.join(command)}")
    process = subprocess.Popen(
        command, cwd=repo_path, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, 
        text=True, encoding='utf-8', errors='ignore'
    )
    for line in iter(process.stdout.readline, ''):
        if line: print(line.strip())
    
    if process.wait() != 0:
        raise RuntimeError("Обучение Silero завершилось с ошибкой.")

    print("\nЛогирование дообученной модели Silero в W&B Artifacts...")
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="upload_silero_model")
    model_artifact = wandb.Artifact(f"silero-finetuned-{run.id}", type="model")
    model_artifact.add_dir(paths_cfg['silero_finetuned_path'])
    run.log_artifact(model_artifact)
    run.finish()

# Рекомендуется запускать на отдельном GPU, установив os.environ["CUDA_VISIBLE_DEVICES"] = "1"
run_stage('silero_training', run_silero_finetune_func)

In [ ]:
def run_yagpt_finetune_func(config):
    llm_cfg = config['yandex_gpt_params']
    paths_cfg = config['paths']
    wandb_cfg = config['wandb']

    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="train_llm_dpo")

    artifact = run.use_artifact('dpo_dataset:latest')
    dataset_path = artifact.download()
    dpo_dataset = Dataset.load_from_disk(dataset_path)

    model_name = llm_cfg['base_model']
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Адаптируем датасет под формат чата, который ожидает DPOTrainer
    def chat_template_format(example):
        example["prompt"] = [{"role": "user", "content": example["prompt"]}]
        example["chosen"] = [{"role": "assistant", "content": example["chosen"]}]
        example["rejected"] = [{"role": "assistant", "content": example["rejected"]}]
        return example

    dpo_dataset = dpo_dataset.map(chat_template_format)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    model.config.use_cache = False
    
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(**llm_cfg['lora_config'])
    model = get_peft_model(model, lora_config)
    
    training_args = TrainingArguments(
        **llm_cfg['dpo_training_args']
    )
    
    dpo_trainer = DPOTrainer(
        model, ref_model=None, args=training_args,
        beta=0.1,
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
    )
    
    print("Запуск DPO fine-tuning...")
    dpo_trainer.train()
    
    adapter_path = Path(paths_cfg['llm_finetuned_adapter_path'])
    adapter_path.mkdir(parents=True, exist_ok=True)
    dpo_trainer.save_model(str(adapter_path))
    print(f"LoRA адаптер сохранен в: {adapter_path}")
    
    model_artifact = wandb.Artifact(f"yagpt-lora-adapter-{run.id}", type="model_adapter")
    model_artifact.add_dir(str(adapter_path))
    run.log_artifact(model_artifact)
    run.finish()

run_stage('llm_finetuning', run_yagpt_finetune_func)

In [ ]:
def merge_lora_adapter_func(config):
    paths_cfg = config['paths']
    wandb_cfg = config['wandb']
    
    print("\nОбъединение LoRA адаптера с базовой моделью...")
    # Загружаем базовую модель
    model = AutoModelForCausalLM.from_pretrained(
        config['yandex_gpt_params']['base_model'],
        torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True
    )
    
    # Применяем LoRA адаптер
    model = AutoPeftModelForCausalLM.from_pretrained(model, paths_cfg['llm_finetuned_adapter_path'])
    model = model.merge_and_unload()
    
    merged_path = Path(paths_cfg['llm_finetuned_merged_path'])
    merged_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(merged_path))
    
    tokenizer = AutoTokenizer.from_pretrained(config['yandex_gpt_params']['base_model'], trust_remote_code=True)
    tokenizer.save_pretrained(str(merged_path))
    
    print(f"Модель успешно объединена и сохранена в: {merged_path}")
    
    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="merge_llm_model")
    model_artifact = wandb.Artifact("yagpt-corrector-merged", type="model")
    model_artifact.add_dir(str(merged_path))
    run.log_artifact(model_artifact)
    run.finish()

run_stage('model_merging', merge_lora_adapter_func)

In [ ]:
def run_evaluation_func(config):
    paths_cfg = config['paths']
    wandb_cfg = config['wandb']

    run = wandb.init(project=wandb_cfg['project'], entity=wandb_cfg['entity'], job_type="final_evaluation")
    
    # Загрузка тестовых данных
    artifact = run.use_artifact('asr_datasets:latest')
    data_dir = Path(artifact.download())
    test_df = pd.read_json(data_dir / "whisper_test.jsonl", lines=True)

    # Загрузка моделей ASR
    print("Загрузка ASR моделей...")
    whisper_model = FasterWhisperModel(paths_cfg['whisper_ct2_path'], device="cuda", compute_type="float16")
    sys.path.insert(0, str(Path(config['silero_params']['repo_path']))); silero_model = torch.jit.load(Path(paths_cfg['silero_finetuned_path']) / "latest.pt", map_location="cuda").eval(); sys.path.pop(0)

    # Инициализация vLLM
    print("Загрузка дообученной LLM для vLLM...")
    llm = LLM(
        model=paths_cfg['llm_finetuned_merged_path'],
        tensor_parallel_size=config['vllm_params']['tensor_parallel_size'],
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(paths_cfg['llm_finetuned_merged_path'])
    sampling_params = SamplingParams(temperature=0.0, max_tokens=512)
    
    results = []
    prompts_for_llm = []
    
    print("Генерация транскрипций ASR моделей...")
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="ASR Inference"):
        audio_path = row['audio']
        
        segments, _ = whisper_model.transcribe(audio_path, language="ru")
        whisper_raw = "".join(s.text for s in segments).strip()
        
        wav, _ = torch.load(audio_path)
        with torch.no_grad():
            silero_raw = silero_model(wav[None, ...].to("cuda"))[0].strip()
        
        chat_prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": config['yandex_gpt_params']['dpo_prompt_template'].format(instruction=whisper_raw)}],
            tokenize=False, add_generation_prompt=True
        )
        prompts_for_llm.append(chat_prompt)
        
        results.append({'whisper_raw': whisper_raw, 'silero_raw': silero_raw, 'reference_norm': normalize_text(row['transcription'])})

    print("Коррекция транскрипций с помощью vLLM...")
    llm_outputs = llm.generate(prompts_for_llm, sampling_params)
    
    for i, output in enumerate(llm_outputs):
        results[i]['llm_corrected_raw'] = output.outputs[0].text.strip()
        
    print("Расчет метрик WER...")
    wer_metric = evaluate.load("wer")
    df_results = pd.DataFrame(results)

    df_results['whisper_norm'] = df_results['whisper_raw'].apply(normalize_text)
    df_results['silero_norm'] = df_results['silero_raw'].apply(normalize_text)
    df_results['rover_norm'] = df_results.apply(lambda row: run_rover_on_pair(row['whisper_norm'], row['silero_norm']), axis=1)
    df_results['llm_corrected_norm'] = df_results['llm_corrected_raw'].apply(normalize_text)
    
    wer_scores = {
        "Whisper": 100 * wer_metric.compute(predictions=df_results["whisper_norm"], references=df_results["reference_norm"]),
        "Silero": 100 * wer_metric.compute(predictions=df_results["silero_norm"], references=df_results["reference_norm"]),
        "Ансамбль (ROVER)": 100 * wer_metric.compute(predictions=df_results["rover_norm"], references=df_results["reference_norm"]),
        "LLM-коррекция (DPO)": 100 * wer_metric.compute(predictions=df_results["llm_corrected_norm"], references=df_results["reference_norm"]),
    }
    
    print("\n" + "="*50)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ КАЧЕСТВА (WER - Word Error Rate)")
    print("="*50)
    for name, wer_score in wer_scores.items():
        print(f"{name:<25}: {wer_score:.2f}%")
    print("="*50)
    
    wandb.log({"evaluation_summary": wer_scores})
    wandb.log({"evaluation_details": wandb.Table(dataframe=df_results[['reference_norm', 'whisper_norm', 'silero_norm', 'rover_norm', 'llm_corrected_raw']])})
    
    results_path = "final_evaluation_results.csv"
    df_results.to_csv(results_path, index=False, encoding='utf-8')
    print(f"Детальные результаты сохранены в {results_path}")
    
    run.finish()

run_stage('evaluation', run_evaluation_func)